# 05 · File Formats & Parquet

CSV is human-readable but
slow, untyped and bulky. **Parquet** is the standard for analytical data:
**columnar**, **compressed**, and **typed**. This notebook shows the difference
and how to read/write Parquet with PyArrow and pandas.

In [ ]:
# ▶ Run this first. Locates the sample data no matter where the kernel starts.
from pathlib import Path

def find_data() -> Path:
    here = Path.cwd()
    for base in (here, *here.parents):
        if (base / 'data' / 'raw').exists():
            return base / 'data'
    raise FileNotFoundError('Run: uv run python data/build_data.py')

DATA = find_data()
RAW = DATA / 'raw'
print('Data directory:', DATA)
print('Raw files:', sorted(p.name for p in RAW.glob('*')))

## Row vs columnar storage

CSV stores data **row by row**. Parquet stores it **column by column**, so a
query touching 2 of 20 columns reads only those 2, and each column compresses
well because similar values sit together. That's why analytics engines love it.

In [ ]:
import pandas as pd
orders = pd.read_csv(RAW / 'orders.csv', parse_dates=['order_ts'])
print(orders.dtypes)
print('rows:', len(orders))

## Write & read Parquet

pandas reads/writes Parquet via PyArrow in one call. Types (dates, ints, floats)
are preserved exactly — no re-parsing strings like CSV.

In [ ]:
out_dir = DATA / 'warehouse'
out_dir.mkdir(parents=True, exist_ok=True)
pq_path = out_dir / 'orders.parquet'

orders.to_parquet(pq_path, engine='pyarrow', compression='snappy')
back = pd.read_parquet(pq_path)
print('types preserved:', back['order_ts'].dtype)   # still datetime64
print('rows:', len(back))

## Size comparison

On real data Parquet is typically several times smaller than CSV. Even on this
tiny sample you can see the effect (and the gap widens with scale).

In [ ]:
csv_path = RAW / 'orders.csv'
csv_bytes = csv_path.stat().st_size
pq_bytes = pq_path.stat().st_size
print(f'CSV:     {csv_bytes:>7} bytes')
print(f'Parquet: {pq_bytes:>7} bytes')
print(f'ratio:   {csv_bytes / pq_bytes:.2f}x')

## Column pruning

Reading only the columns you need is a big speed win with columnar files. Pass
`columns=` to skip the rest entirely at the storage layer.

In [ ]:
slim = pd.read_parquet(pq_path, columns=['order_id', 'amount'])
print(slim.columns.tolist())
print('total:', round(slim['amount'].sum(), 2))

## Partitioning

Splitting output into folders by a key (e.g. `status=`) lets engines skip whole
partitions — **partition pruning**. PyArrow writes a partitioned dataset; the
partition value becomes a folder, not a stored column.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.Table.from_pandas(orders)
dataset_dir = out_dir / 'orders_by_status'
pq.write_to_dataset(table, root_path=str(dataset_dir), partition_cols=['status'])

from pathlib import Path
print('partitions written:')
for p in sorted(Path(dataset_dir).glob('status=*')):
    print(' ', p.name)

# reading back can filter to one partition cheaply
completed = pd.read_parquet(dataset_dir, filters=[('status', '==', 'completed')])
print('completed rows read:', len(completed))

## Inspecting metadata without reading data

Parquet stores a schema and row-group statistics in its footer. You can read the
schema and row count **without** loading the data — handy for validation.

In [ ]:
import pyarrow.parquet as pq
meta = pq.read_metadata(pq_path)
print('num_rows:', meta.num_rows)
print('num_columns:', meta.num_columns)
print('schema:\n', pq.read_schema(pq_path))

### Recap

Parquet is columnar, compressed and typed — the analytics default; pandas
reads/writes it via PyArrow with types preserved; `columns=` prunes columns and
partitioning prunes rows; footer metadata gives schema/row-count without a full
read. Next: the capstone ETL pipeline that ties it all together.